# This is my FYP model detection, improved version of P1 Defence. 
# By Mahnoor Bibi 

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json

ROOT = Path.cwd()

candidate_a = ROOT / "crime.v8i.yolov8"
candidate_b = ROOT / "models" / "weaponDetectionModel" / "crime.v8i.yolov8"

if candidate_a.exists():
    DATASET_ROOT = candidate_a
elif candidate_b.exists():
    DATASET_ROOT = candidate_b
else:
    DATASET_ROOT = candidate_a

SPLITS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG", ".PNG"}
CLASS_NAMES = {0: "criminal", 1: "person", 2: "weapon"}

print("Notebook CWD:", ROOT)
print("Dataset root:", DATASET_ROOT)
print("Exists:", DATASET_ROOT.exists())

# Count trainingi and validation data elements 

In [ ]:
split_inventory = {}

for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    images = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix in IMAGE_EXTS]) if img_dir.exists() else []
    labels = sorted(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

    split_inventory[split] = {
        "images": images,
        "labels": labels,
        "image_count": len(images),
        "label_count": len(labels),
    }

inventory_view = {
    s: {
        "image_count": split_inventory[s]["image_count"],
        "label_count": split_inventory[s]["label_count"],
    }
    for s in SPLITS
}

print(json.dumps(inventory_view, indent=2))
print("Total images:", sum(v["image_count"] for v in inventory_view.values()))

In [ ]:
box_counts = Counter()
image_presence = Counter()
malformed_rows = []
invalid_class_rows = []
invalid_range_rows = []
empty_label_files = []

for split in SPLITS:
    for label_path in split_inventory[split]["labels"]:
        lines = label_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()

        if not lines:
            empty_label_files.append((split, str(label_path)))
            continue

        classes_in_image = set()

        for line_no, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                malformed_rows.append((split, str(label_path), line_no, line))
                continue

            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except Exception:
                malformed_rows.append((split, str(label_path), line_no, line))
                continue

            if cls not in CLASS_NAMES:
                invalid_class_rows.append((split, str(label_path), line_no, cls))
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                invalid_range_rows.append((split, str(label_path), line_no, (x, y, w, h)))
                continue

            box_counts[cls] += 1
            classes_in_image.add(cls)

        for cls in classes_in_image:
            image_presence[cls] += 1

summary_counts = {
    "box_counts": {CLASS_NAMES[k]: box_counts[k] for k in sorted(CLASS_NAMES)},
    "image_presence": {CLASS_NAMES[k]: image_presence[k] for k in sorted(CLASS_NAMES)},
    "empty_label_files": len(empty_label_files),
    "malformed_rows": len(malformed_rows),
    "invalid_class_rows": len(invalid_class_rows),
    "invalid_range_rows": len(invalid_range_rows),
}

print(json.dumps(summary_counts, indent=2))